# Кулонівський інтеграл $J$ для $1s^2$-стану гелію — обчислення в SymPy

Цей ноутбук повторює аналітичний вивід кулонівського інтеграла

$$
J = \iint \frac{|\Psi_{100}(\mathbf r_1)|^2\,|\Psi_{100}(\mathbf r_2)|^2}{|\mathbf r_1-\mathbf r_2|}\,dV_1\,dV_2
$$

крок за кроком **символьно**, засобами `sympy`, включно з `sympy.physics.hydrogen` для власне
водневоподібних хвильових функцій. У кінці отримуємо відомий результат

$$
J = \frac{5}{8}Z .
$$

In [13]:
import sympy as sp
from sympy import (symbols, sqrt, sin, cos, pi, exp, oo, integrate,
                    simplify, Piecewise, factor, nsimplify, Abs, latex)
from sympy.physics.hydrogen import R_nl, Psi_nlm, E_nl

sp.init_printing(use_latex='mathjax')


## 1. Хвильова функція $1s$ через `sympy.physics.hydrogen`

Замість того, щоб вручну виписувати $\Psi_{100}(r)=\sqrt{Z^3/\pi}\,e^{-Zr}$, візьмемо її з
готового модуля `sympy.physics.hydrogen`:

- `R_nl(n, l, r, Z)` — радіальна частина $R_{nl}(r)$ (атомні одиниці, $a_0=1$);
- `Psi_nlm(n, l, m, r, phi, theta, Z)` — повна нормована хвильова функція $\Psi_{nlm}(\mathbf r)$,
  що вже включає сферичну гармоніку $Y_{lm}$.


In [2]:
r, r1, r2, theta, theta1, theta2, phi, phi1, phi2, Z = symbols(
    'r r1 r2 theta theta1 theta2 phi phi1 phi2 Z', positive=True)

R10 = R_nl(1, 0, r, Z)
R10


   3/2  -Z⋅r
2⋅Z   ⋅ℯ    

In [14]:
Psi100 = Psi_nlm(1, 0, 0, r, phi, theta, Z)
Psi100 = simplify(Psi100)
Psi100


 3/2  -Z⋅r
Z   ⋅ℯ    
──────────
    √π    

In [15]:
density = simplify(Abs(Psi100)**2)
density   #  |Psi_100(r)|^2 = Z**3/pi * exp(-2*Z*r)


 3  -2⋅Z⋅r
Z ⋅ℯ      
──────────
    π     

## 2. Геометрія: інтегрування по $\theta_2$

Обираємо вісь $z$ вздовж $\mathbf r_1$, тому

$$
|\mathbf r_1-\mathbf r_2| = \sqrt{r_1^2+r_2^2-2r_1r_2\cos\theta_2}.
$$

Беремо кутовий інтеграл

$$
\int_0^\pi \frac{\sin\theta_2\,d\theta_2}{\sqrt{r_1^2+r_2^2-2r_1r_2\cos\theta_2}}
$$

прямо через `sympy.integrate` — жодних «ручних» тотожностей вводити не потрібно, SymPy сам
бере цей інтеграл.


In [16]:
ang_kernel = sin(theta2) / sqrt(r1**2 + r2**2 - 2*r1*r2*cos(theta2))

angular_raw = integrate(ang_kernel, (theta2, 0, pi))
angular_raw


     _____________________      _____________________
    ╱   2               2      ╱   2               2 
  ╲╱  r₁  - 2⋅r₁⋅r₂ + r₂     ╲╱  r₁  + 2⋅r₁⋅r₂ + r₂  
- ──────────────────────── + ────────────────────────
           r₁⋅r₂                      r₁⋅r₂          

SymPy повертає вираз через $\sqrt{(r_1+r_2)^2}$ та $\sqrt{(r_1-r_2)^2}$. Перший корінь
завжди дорівнює $r_1+r_2$ (обидва додатні), а другий — це $|r_1-r_2|$, знак якого залежить від
того, яке з $r_1,r_2$ більше. Запишемо це явно через `Piecewise`, як і в конспекті:

$$
\int_0^\pi \frac{\sin\theta_2\,d\theta_2}{\sqrt{r_1^2+r_2^2-2r_1r_2\cos\theta_2}}
= \begin{cases}\dfrac{2}{r_1}, & r_1>r_2\\[4pt] \dfrac{2}{r_2}, & r_2>r_1\end{cases}
= \frac{2}{\max(r_1,r_2)}.
$$


In [18]:
angular_piecewise = Piecewise((2/r1, r1 > r2), (2/r2, True))
angular_piecewise

⎧2              
⎪──  for r₁ > r₂
⎪r₁             
⎨               
⎪2              
⎪──   otherwise 
⎩r₂             

## 3. Інтегрування по $r_2$ (і $\varphi_2$)

Тепер беремо радіальний інтеграл по $r_2\in(0,\infty)$, розбитий на дві ділянки $r_2<r_1$ та
$r_2>r_1$ відповідно до знайденого `Piecewise`, плюс тривіальний інтеграл по $\varphi_2$
(дає множник $2\pi$) і множник густини $|\Psi_{100}(r_2)|^2=\dfrac{Z^3}{\pi}e^{-2Zr_2}$.


In [19]:
inner_r2_lt = integrate(r2**2 * exp(-2*Z*r2) * (2/r1), (r2, 0, r1))
inner_r2_gt = integrate(r2**2 * exp(-2*Z*r2) * (2/r2), (r2, r1, oo))

inner_r2 = simplify(inner_r2_lt + inner_r2_gt)
inner_r2


⎛         2⋅Z⋅r₁    ⎞  -2⋅Z⋅r₁
⎝-Z⋅r₁ + ℯ       - 1⎠⋅ℯ       
──────────────────────────────
              3               
           2⋅Z ⋅r₁            

In [20]:
# множимо на 2*pi (інтеграл по phi2) та на Z**3/pi (нормування |Psi_100(r2)|^2)
electron2_integral = simplify(2*pi * (Z**3/pi) * inner_r2)
electron2_integral


⎛         2⋅Z⋅r₁    ⎞  -2⋅Z⋅r₁
⎝-Z⋅r₁ + ℯ       - 1⎠⋅ℯ       
──────────────────────────────
              r₁              

Це саме вираз (7) із конспекту:

$$
\int dV_2\,\frac{|\Psi_{100}(\mathbf r_2)|^2}{|\mathbf r_1-\mathbf r_2|}
= \frac{\bigl(-Zr_1+e^{2Zr_1}-1\bigr)e^{-2Zr_1}}{r_1}.
$$


## 4. Останнє інтегрування по $\mathbf r_1$

Залишається взяти $dV_1=r_1^2\sin\theta_1\,dr_1\,d\theta_1\,d\varphi_1$; кутова частина дає
множник $4\pi$, а по $r_1$ бере вже звичайний однорідний інтеграл $\int_0^\infty r_1^n
e^{-ar_1}dr_1$.


In [21]:
J_integrand = 4*pi * r1**2 * (Z**3/pi) * exp(-2*Z*r1) * electron2_integral
J_integrand = simplify(J_integrand)
J_integrand


   3    ⎛         2⋅Z⋅r₁    ⎞  -4⋅Z⋅r₁
4⋅Z ⋅r₁⋅⎝-Z⋅r₁ + ℯ       - 1⎠⋅ℯ       

In [22]:
J = integrate(J_integrand, (r1, 0, oo))
J = simplify(J)
J


5⋅Z
───
 8 

## 5. Усе одразу «за один виклик» — повний 6-вимірний інтеграл

Для порівняння: якщо дати SymPy взяти інтеграл одразу по $\theta_1,\varphi_1$ (тривіально,
бо підінтегральний вираз від них не залежить) і скласти результат із кроків 2–4 в одну
функцію, отримаємо той самий результат. Наведемо це компактно.


In [13]:
def coulomb_integral_J(Zsym=Z):
    r1, r2, theta2 = symbols('r1 r2 theta2', positive=True)

    # 1) кутовий інтеграл по theta2
    ang = integrate(sin(theta2)/sqrt(r1**2+r2**2-2*r1*r2*cos(theta2)), (theta2, 0, pi))
    ang_piecewise = Piecewise((2/r1, r1 > r2), (2/r2, True))

    # 2) радіальний інтеграл по r2 (+ phi2 -> 2*pi, + густина Z**3/pi)
    inner = integrate(r2**2*exp(-2*Zsym*r2)*(2/r1), (r2, 0, r1)) \
          + integrate(r2**2*exp(-2*Zsym*r2)*(2/r2), (r2, r1, oo))
    e2 = 2*pi*(Zsym**3/pi)*inner

    # 3) інтеграл по r1 (+ theta1,phi1 -> 4*pi, + густина Z**3/pi)
    integrand = 4*pi*r1**2*(Zsym**3/pi)*exp(-2*Zsym*r1)*e2
    return simplify(integrate(simplify(integrand), (r1, 0, oo)))

coulomb_integral_J()


5⋅Z
───
 8 